In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
gdf = gpd.read_file('data/processed/Collision_Processed.geojson')
gdf

In [ ]:
grid = gdf.groupby(['lat_bin', 'lon_bin']).agg(
    collision_count=('COLDETKEY','count'),
    mean_severity=('SEVERITY', 'mean'),
    total_injuries=('INJURIES', 'sum'),
    total_serious=('SERIOUSINJURIES', 'sum'),
    total_fatalities=('FATALITIES','sum'),
    avg_vehicle_count=('VEHCOUNT', 'mean'),
    avg_pedestrian_count=('TOTAL_PED', 'mean')
).reset_index()


In [ ]:
scaler = StandardScaler()
grid[['c_density_norm', 's_severity_norm']] = scaler.fit_transform(grid[['collision_count', 'mean_severity']])
grid['risk_score'] = 0.6 * grid['c_density_norm'] + 0.4 * grid['s_severity_norm']
grid['risk_label'] = pd.qcut(grid['risk_score'], q=3, labels=['low risk', 'medium risk', 'high risk'])
grid.get(['risk_score', 'risk_label'])

In [ ]:
feature_cols = ['collision_count', 'mean_severity', 'total_injuries', 'total_serious', 
                'total_fatalities', 'avg_vehicle_count', 'avg_pedestrian_count']
le = LabelEncoder()

X = grid[feature_cols]
y = grid['risk_label']
y = le.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model = DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv)

print('CV accuracy scores:', np.round(cv_scores, 3))
print('Mean CV scores:', np.round(cv_scores.mean(), 3))

In [ ]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Test accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification report:\n', classification_report(y_test, y_pred, target_names=le.classes_))

fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi.plot(kind='barh')
plt.title('Feature Importance')
plt.show()

class_names = np.unique(np.concatenate((y_pred, y_test)))
cm_collision = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm_collision, display_labels=class_names).plot()
plt.title('Confusion Matrix for Collision Risk Test & Prediction Set')
plt.show()

In [ ]:
plt.figure(figsize=(200, 40))
plot_tree(model, feature_names=feature_cols, class_names=le.classes_, 
          filled=True, fontsize=35)
plt.title('Decision Tree for Zone Risk Classification')
plt.show()

In [ ]:
# Extract lat/lon from bins for mapping
def parse_interval(s):
    s = s.strip('()[]')
    left, right = s.split(', ')
    return (float(left) + float(right)) / 2

grid['lat'] = grid['lat_bin'].apply(parse_interval)
grid['lon'] = grid['lon_bin'].apply(parse_interval)

In [ ]:
risk_map = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid.lon, grid.lat), crs=gdf.crs)

fig, ax = plt.subplots(1,1,figsize=(24,20))
risk_map.plot(column='risk_label', categorical=True, legend=True,
              cmap='flare', ax=ax, markersize=40, alpha=0.8)
ax.set_title('Grid Cell Risk Label')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()